# CodeMix-T — Phase 4: Ablation Studies & Cluster Training

Covers:
1. Ablation experiment configs (with/without LangID, model size, joint vs separate)
2. SLURM job script generator for university cluster
3. Automated ablation runner
4. Results comparison table
5. Learning curve plotting (WandB)
6. Best model selection

## Cell 1 — Ablation Configurations

In [ ]:
# Each ablation removes or modifies ONE component.
# This gives you clean evidence for your paper about what each part contributes.

ABLATION_CONFIGS = {

    # ── EXPERIMENT 1: Full model (your main result) ──
    'full_base': {
        'd_model': 512, 'd_ff': 2048, 'num_heads': 8,
        'num_enc_layers': 6, 'num_dec_layers': 6,
        'lang_embed_dim': 64,
        'use_lang_id': True,          # Language-ID embeddings ON
        'train_languages': 'both',    # Train on Hinglish + Tanglish together
        'description': 'Full CodeMix-T with LangID embeddings'
    },

    # ── EXPERIMENT 2: No LangID embeddings ──
    # Tests: how much do Language-ID embeddings contribute?
    # Paper claim: removing them should hurt BLEU score
    'no_lang_id': {
        'd_model': 512, 'd_ff': 2048, 'num_heads': 8,
        'num_enc_layers': 6, 'num_dec_layers': 6,
        'lang_embed_dim': 0,
        'use_lang_id': False,         # Language-ID embeddings OFF
        'train_languages': 'both',
        'description': 'Ablation: no Language-ID embeddings'
    },

    # ── EXPERIMENT 3: Small model ──
    # Tests: impact of model size on code-mixed translation
    'small_lang_id': {
        'd_model': 256, 'd_ff': 1024, 'num_heads': 4,
        'num_enc_layers': 4, 'num_dec_layers': 4,
        'lang_embed_dim': 32,
        'use_lang_id': True,
        'train_languages': 'both',
        'description': 'Small model (256 dim) with LangID'
    },

    # ── EXPERIMENT 4: Hinglish only ──
    # Tests: does joint training help or hurt per-language performance?
    'hinglish_only': {
        'd_model': 512, 'd_ff': 2048, 'num_heads': 8,
        'num_enc_layers': 6, 'num_dec_layers': 6,
        'lang_embed_dim': 64,
        'use_lang_id': True,
        'train_languages': 'hinglish',
        'description': 'Hinglish-only training'
    },

    # ── EXPERIMENT 5: Tanglish only ──
    'tanglish_only': {
        'd_model': 512, 'd_ff': 2048, 'num_heads': 8,
        'num_enc_layers': 6, 'num_dec_layers': 6,
        'lang_embed_dim': 64,
        'use_lang_id': True,
        'train_languages': 'tanglish',
        'description': 'Tanglish-only training'
    },
}

print(f'{len(ABLATION_CONFIGS)} ablation experiments defined:')
for k, v in ABLATION_CONFIGS.items():
    print(f'  {k:20s}: {v["description"]}')

## Cell 2 — Modified CodeMixEmbedding Supporting Ablation

In [ ]:
# Modified embedding that can disable Language-ID (for ablation experiments)
# Paste this INSTEAD of the original CodeMixEmbedding in Phase 2 when running ablations

class CodeMixEmbeddingAblation(nn.Module):
    """
    Same as CodeMixEmbedding but with optional LangID bypass.
    Set use_lang_id=False to run the no-LangID ablation.
    """
    def __init__(self, cfg, use_lang_id: bool = True):
        super().__init__()
        self.use_lang_id = use_lang_id
        self.token_embed = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.pos_encoding = PositionalEncoding(cfg.d_model, cfg.max_seq_len, cfg.dropout)
        self.dropout = nn.Dropout(cfg.dropout)
        self.scale = math.sqrt(cfg.d_model)

        if use_lang_id and cfg.lang_embed_dim > 0:
            self.lang_embed = nn.Embedding(cfg.num_languages, cfg.lang_embed_dim)
            self.lang_proj  = nn.Linear(cfg.lang_embed_dim, cfg.d_model, bias=False)
        else:
            self.lang_embed = None
            self.lang_proj  = None

    def forward(self, token_ids, lang_ids):
        tok = self.token_embed(token_ids) * self.scale
        if self.use_lang_id and self.lang_embed is not None:
            lang = self.lang_proj(self.lang_embed(lang_ids))
            tok  = tok + lang
        return self.pos_encoding(tok)


print('AblationEmbedding defined.')

## Cell 3 — Ablation Runner

In [ ]:
import wandb, json, os, copy

ABLATION_RESULTS = {}

def run_ablation(exp_name: str, exp_cfg: dict, epochs: int = 10):
    """
    Train one ablation variant and return best val loss.
    Uses fewer epochs (10) since we only need relative comparison.
    """
    print(f'\n{"="*55}')
    print(f'Running ablation: {exp_name}')
    print(f'  {exp_cfg["description"]}')
    print(f'{"="*55}')

    # Build config for this ablation
    abl_cfg = CodeMixTConfig(
        vocab_size      = cfg.vocab_size,
        pad_id          = cfg.pad_id,
        bos_id          = cfg.bos_id,
        eos_id          = cfg.eos_id,
        d_model         = exp_cfg['d_model'],
        d_ff            = exp_cfg['d_ff'],
        num_heads       = exp_cfg['num_heads'],
        num_enc_layers  = exp_cfg['num_enc_layers'],
        num_dec_layers  = exp_cfg['num_dec_layers'],
        lang_embed_dim  = exp_cfg['lang_embed_dim'],
        max_seq_len     = cfg.max_seq_len,
        dropout         = cfg.dropout,
    )

    # Build model (use ablation embedding)
    abl_model = CodeMixT(abl_cfg).to(device)
    # Swap embedding if ablation disables LangID
    abl_model.encoder.embedding = CodeMixEmbeddingAblation(
        abl_cfg, use_lang_id=exp_cfg['use_lang_id']
    ).to(device)

    abl_optimizer = torch.optim.Adam(abl_model.parameters(), lr=1.0, betas=(0.9,0.98), eps=1e-9)
    abl_scheduler = WarmupInvSqrtScheduler(abl_optimizer, abl_cfg.d_model)
    abl_criterion = LabelSmoothingLoss(abl_cfg.vocab_size, abl_cfg.pad_id)

    # Filter dataloader by language if needed
    if exp_cfg['train_languages'] == 'hinglish':
        subset = [i for i, s in enumerate(train_ds.samples) if 'hindi' in str(s).lower() or 'hinglish' in str(s).lower()]
        abl_loader = DataLoader(torch.utils.data.Subset(train_ds, subset),
                                batch_size=BATCH_SIZE, shuffle=True,
                                collate_fn=lambda b: collate_fn(b, abl_cfg.pad_id))
    elif exp_cfg['train_languages'] == 'tanglish':
        subset = [i for i, s in enumerate(train_ds.samples) if 'tanglish' in str(s).lower()]
        abl_loader = DataLoader(torch.utils.data.Subset(train_ds, subset),
                                batch_size=BATCH_SIZE, shuffle=True,
                                collate_fn=lambda b: collate_fn(b, abl_cfg.pad_id))
    else:
        abl_loader = train_loader

    wandb.init(project='codemix-t-ablations', name=exp_name,
               config={**exp_cfg, 'epochs': epochs}, reinit=True)

    best_val = float('inf')
    global_step = 0

    for epoch in range(epochs):
        abl_model.train()
        for batch in tqdm(abl_loader, desc=f'  Epoch {epoch+1}', leave=False):
            loss, lr = train_step(abl_model, batch, abl_optimizer, abl_scheduler, abl_criterion, device)
            global_step += 1
            if global_step % 100 == 0:
                wandb.log({'train/loss': loss, 'train/lr': lr, 'step': global_step})

        val_loss, val_ppl = val_step(abl_model, val_loader, abl_criterion, device)
        wandb.log({'val/loss': val_loss, 'val/ppl': val_ppl, 'epoch': epoch+1})
        if val_loss < best_val:
            best_val = val_loss
            torch.save(abl_model.state_dict(),
                       f'{DRIVE_DIR}/checkpoints/ablation_{exp_name}_best.pt')

        print(f'  Epoch {epoch+1}: val_loss={val_loss:.4f}, val_ppl={val_ppl:.2f}')

    wandb.finish()
    ABLATION_RESULTS[exp_name] = {
        'description': exp_cfg['description'],
        'best_val_loss': best_val,
        'params': abl_model.count_parameters()
    }
    print(f'  Best val_loss = {best_val:.4f}')
    del abl_model
    torch.cuda.empty_cache()
    return best_val


# Run all ablations (comment out experiments to skip)
for exp_name, exp_cfg in ABLATION_CONFIGS.items():
    run_ablation(exp_name, exp_cfg, epochs=10)

## Cell 4 — Results Comparison Table

In [ ]:
import pandas as pd

if ABLATION_RESULTS:
    results_df = pd.DataFrame([
        {
            'Experiment': k,
            'Description': v['description'],
            'Val Loss': round(v['best_val_loss'], 4),
            'Perplexity': round(math.exp(min(v['best_val_loss'], 20)), 2),
            'Params (M)': round(v['params'] / 1e6, 1)
        }
        for k, v in ABLATION_RESULTS.items()
    ])
    results_df = results_df.sort_values('Val Loss')
    print('\nAblation Results:')
    print(results_df.to_string(index=False))

    # Save results
    results_df.to_csv(f'{DRIVE_DIR}/ablation_results.csv', index=False)
    print(f'\nSaved to {DRIVE_DIR}/ablation_results.csv')
else:
    print('No ablation results yet. Run Cell 3 first.')

## Cell 5 — SLURM Job Script Generator (for university cluster)

In [ ]:
# Generate SLURM scripts for each ablation experiment
# Upload these to your university cluster to run experiments in parallel

SLURM_TEMPLATE = '''#!/bin/bash
#SBATCH --job-name=codemix_{exp_name}
#SBATCH --output=logs/codemix_{exp_name}_%j.out
#SBATCH --error=logs/codemix_{exp_name}_%j.err
#SBATCH --time=12:00:00
#SBATCH --gres=gpu:1
#SBATCH --mem=32G
#SBATCH --cpus-per-task=4
#SBATCH --partition=gpu

module load python/3.10
module load cuda/11.8
source ~/codemix_env/bin/activate

cd ~/CodeMixT

python train.py \\
  --exp_name {exp_name} \\
  --d_model {d_model} \\
  --d_ff {d_ff} \\
  --num_heads {num_heads} \\
  --enc_layers {num_enc_layers} \\
  --dec_layers {num_dec_layers} \\
  --lang_embed_dim {lang_embed_dim} \\
  --use_lang_id {use_lang_id} \\
  --languages {train_languages} \\
  --epochs 30 \\
  --batch_size 64 \\
  --data_dir /scratch/$USER/codemix_data \\
  --ckpt_dir /scratch/$USER/codemix_ckpts
'''

os.makedirs('slurm_scripts', exist_ok=True)

for exp_name, exp_cfg in ABLATION_CONFIGS.items():
    script = SLURM_TEMPLATE.format(
        exp_name        = exp_name,
        d_model         = exp_cfg['d_model'],
        d_ff            = exp_cfg['d_ff'],
        num_heads       = exp_cfg['num_heads'],
        num_enc_layers  = exp_cfg['num_enc_layers'],
        num_dec_layers  = exp_cfg['num_dec_layers'],
        lang_embed_dim  = exp_cfg['lang_embed_dim'],
        use_lang_id     = int(exp_cfg['use_lang_id']),
        train_languages = exp_cfg['train_languages'],
    )
    path = f'slurm_scripts/run_{exp_name}.sh'
    with open(path, 'w') as f:
        f.write(script)
    print(f'Written: {path}')

print('\nTo submit all jobs on the cluster:')
print('  for f in slurm_scripts/*.sh; do sbatch $f; done')